# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


**Finding 1 (page 27, ML Appendix, Feature Importance for Health Score)**

the paper says avg position is the number 1 feature at 43% importance, then impressions at 32%, then scroll depth at 15%, when the random forest tries to predict health score.

my question, isnt half of this expected since health score literally already contains position, impressions and scroll depth inside its own formula (page 5, health score = impressions 30pts plus position 30pts plus ctr 20pts plus scroll depth 20pts). so the model is not really discovering something new, its just kind of finding its way back to the formula it was built from. the paper does say this is descriptive not causal which is good, but the question i would ask is did anyone try training the model without those 3 features just to see if it still explains a good chunk of health score using only the leftover stuff like clicks, sessions, content age, word count. that one test would show how much of that 43 percent is a real pattern vs just arithmetic.

**Finding 2 (page 9, Finding 4, The Freshness Multiplier)**

the paper says pages older than 365 days that got refreshed in the last 30 days show a 3.2x health boost and 57x more impressions.

my question here is about who picked the pages to refresh. were these random old pages or did an editor pick ones that still had some traffic or backlinks left, ones that felt worth saving. if it was the second thing then some of that 3.2x/57x number is just because the pages were already decent to begin with, not only because they got refreshed. it would help to see this same comparison but only for old pages that had similar traffic before the refresh happened, so its apples to apples.


In [1]:
# just writing down the 2 findings and my questions again in code so its easy to see, nothing to calculate here
paper_findings = [
   {"finding": "feature importance for health score (p.27)", "question": "is 43 percent real or just the formula talking to itself"},
   {"finding": "freshness multiplier 3.2x/57x (p.9)", "question": "were refreshed pages picked because they already looked ok"},
]

for f in paper_findings:
   print(f["finding"])
   print("  ->", f["question"])


feature importance for health score (p.27)
  -> is 43 percent real or just the formula talking to itself
freshness multiplier 3.2x/57x (p.9)
  -> were refreshed pages picked because they already looked ok


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


In [2]:
# --- Colab setup: clone the repo and regenerate the processed data (gitignored, pipeline-built) ---
import os

REPO_URL = "https://github.com/muzammil-12345/flyrank-ml-internship.git"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.isdir(os.path.join(REPO_DIR, "scripts")):
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}/scripts
# data/processed/ is gitignored, these 2 scripts build it again from data/raw
!python3 01_prepare_features.py
!python3 02_baseline_score.py

%cd {REPO_DIR}/work/notebooks
print("Working directory set to:", os.getcwd())


/content/flyrank-ml-internship/scripts
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340
/content/flyrank-ml-internship/work/notebooks
Working directory set to: /content/flyrank-ml-internship/work/notebooks


In [3]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# find the repo root no matter where this notebook gets run from
def find_repo_root(start='.'):
   path = os.path.abspath(start)
   for i in range(6):
      if os.path.isdir(os.path.join(path, 'scripts')) and os.path.isdir(os.path.join(path, 'data')):
         return path
      path = os.path.dirname(path)
   raise FileNotFoundError("couldnt find repo root, clone the repo first")

REPO_ROOT = find_repo_root()
sys.path.insert(0, os.path.join(REPO_ROOT, 'scripts'))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

RANDOM_STATE = 42
feat = pd.read_csv(os.path.join(REPO_ROOT, 'data/processed/refresh_feature_vector.csv'))
target = feat['is_declining_label']
idx = np.arange(len(feat))

numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in feat.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in feat.columns]
numeric_frame = feat[numeric_features].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
cat_frame = feat[categorical_features].fillna('unknown').astype(str)
encoded = pd.get_dummies(cat_frame, prefix=categorical_features, dummy_na=False, dtype=float)
X = pd.concat([numeric_frame.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)
y = target.reset_index(drop=True)

# trend_direction and trend_pct stay out, they basically make the label (label trap from week 3)
print("feature matrix:", X.shape[1], "columns,", len(feat), "rows")


feature matrix: 52 columns, 30000 rows


In [4]:
def run_split(train_idx, test_idx, label):
   model = RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25,
                                   n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
   model.fit(X.iloc[train_idx], y.iloc[train_idx])
   proba = model.predict_proba(X.iloc[test_idx])[:, 1]
   y_test = y.iloc[test_idx]
   print(label)
   print("  rows in test:", len(test_idx))
   print("  roc auc:", round(roc_auc_score(y_test, proba), 3))
   print("  precision@50:", round(precision_at_k(y_test, proba, 50), 3))
   print("  base rate:", round(y_test.mean(), 3))
   print()

# BEFORE, just a normal random row split, the kind you'd do without thinking twice
rng1 = np.random.default_rng(RANDOM_STATE)
shuffled_rows = rng1.permutation(idx)
n_test = int(round(len(idx) * 0.2))
test_idx_random = shuffled_rows[:n_test]
train_idx_random = shuffled_rows[n_test:]
run_split(train_idx_random, test_idx_random, "BEFORE - random row split")

# AFTER, grouped by client_id so the same client cant be in both train and test
# same split logic as week 5 model
rng2 = np.random.default_rng(RANDOM_STATE)
client_series = feat['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
shuffled_clients = rng2.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients).to_numpy()
train_idx_grouped = idx[~test_mask]
test_idx_grouped = idx[test_mask]
run_split(train_idx_grouped, test_idx_grouped, "AFTER - grouped by client_id split")

overlap = set(feat.iloc[train_idx_grouped]['client_id']) & set(feat.iloc[test_idx_grouped]['client_id'])
print("clients overlapping train and test:", len(overlap), "(should be 0)")


BEFORE - random row split
  rows in test: 6000
  roc auc: 0.771
  precision@50: 1.0
  base rate: 0.541

AFTER - grouped by client_id split
  rows in test: 2325
  roc auc: 0.75
  precision@50: 0.74
  base rate: 0.391

clients overlapping train and test: 0 (should be 0)


**what i see here.** the random split gets a precision@50 of basically 1.000, like almost a perfect score, but the honest grouped split (same one week 5 already used) only gets 0.740. roc auc barely moves (0.771 vs 0.750) but precision@50 drops a lot. i think whats happening is the random split lets the model see a few rows from every single client while training, so it kind of memorizes stuff specific to a client instead of learning a real decline pattern, and that memorized stuff doesnt help when the client is totally new in the grouped test. so the honest number to actually report is 0.740, not the 1.000 one, even tho 1.000 looks way better on paper.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


In [5]:
# checking the leakage stuff again but on the split from section 2

def build_X(extra_numeric=None, extra_categorical=None):
   nf = numeric_features + (extra_numeric or [])
   cf = categorical_features + (extra_categorical or [])
   numeric_part = feat[nf].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
   cat_part = feat[cf].fillna('unknown').astype(str)
   encoded_part = pd.get_dummies(cat_part, prefix=cf, dummy_na=False, dtype=float)
   return pd.concat([numeric_part.reset_index(drop=True), encoded_part.reset_index(drop=True)], axis=1)

X_clean = build_X()
# adding the known bad ones back on purpose just to see the score explode
X_leaky = build_X(extra_numeric=['trend_pct'], extra_categorical=['trend_direction'])

def run_on(X_variant, label):
   model = RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25,
                                   n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
   model.fit(X_variant.iloc[train_idx_grouped], y.iloc[train_idx_grouped])
   proba = model.predict_proba(X_variant.iloc[test_idx_grouped])[:, 1]
   y_test = y.iloc[test_idx_grouped]
   print(label)
   print("  roc auc:", round(roc_auc_score(y_test, proba), 3))
   print("  precision@50:", round(precision_at_k(y_test, proba, 50), 3))
   print()
   return model

run_on(X_clean, "without the suspects (this is my real feature set)")
run_on(X_leaky, "with trend_pct + trend_direction added back (leaky on purpose)")


without the suspects (this is my real feature set)
  roc auc: 0.75
  precision@50: 0.74

with trend_pct + trend_direction added back (leaky on purpose)
  roc auc: 1.0
  precision@50: 1.0



RandomForestClassifier(class_weight='balanced_subsample', max_depth=10,
                       min_samples_leaf=25, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [6]:
# also checking base rate and top feature importance doesnt look sus
clean_model = RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25,
                                      n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
clean_model.fit(X_clean.iloc[train_idx_grouped], y.iloc[train_idx_grouped])
importances = pd.Series(clean_model.feature_importances_, index=X_clean.columns).sort_values(ascending=False)

print("base rate on grouped test:", round(y.iloc[test_idx_grouped].mean(), 3))
print("top feature share:", round(importances.iloc[0], 3))
importances.head(6)


base rate on grouped test: 0.391
top feature share: 0.135


,0
days_with_impressions,0.134951
log_impressions_90d,0.129377
avg_position,0.109203
content_age_days,0.092048
char_count,0.038676
age_tier_365+,0.036847


**what the audit shows.** when i put trend_pct and trend_direction back in, both roc auc and precision@50 jump straight to 1.000. thats basically the model cheating, since those 2 columns are literally what the label is made from, so they stay out for good, same as week 3 and week 5. on the clean model the top feature (days_with_impressions) is only about 13 percent of the importance, its not like one single feature is doing everything, which is a decent sign nothing is leaking quietly. and the base rate (0.391) is way below the 0.740 precision@50, so the model really is doing something, its not just an easy label making it look good.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


In [7]:
original_claim = "random forest wins on every metric, it scores 0.740 vs the baseline's 0.240, it predicts which pages are declining"

problem_with_it = "'it predicts which pages are declining' sounds like its 100 percent sure about every single page, but really its just right more often than random guessing, not a guarantee for any one page"

rewritten_claim = (
   "under a client-grouped holdout split, the random forest model ranks pages by estimated decline risk, "
   "with a measured precision@50 of 0.740 (base rate 0.391). that means about 74 percent of the top 50 flagged "
   "pages were actually declining in this holdout sample. this is decision-support to help a reviewer pick which "
   "pages to check first, not a guaranteed answer for any single page."
)

print("original:", original_claim)
print()
print("whats wrong:", problem_with_it)
print()
print("rewritten:", rewritten_claim)


original: random forest wins on every metric, it scores 0.740 vs the baseline's 0.240, it predicts which pages are declining

whats wrong: 'it predicts which pages are declining' sounds like its 100 percent sure about every single page, but really its just right more often than random guessing, not a guarantee for any one page

rewritten: under a client-grouped holdout split, the random forest model ranks pages by estimated decline risk, with a measured precision@50 of 0.740 (base rate 0.391). that means about 74 percent of the top 50 flagged pages were actually declining in this holdout sample. this is decision-support to help a reviewer pick which pages to check first, not a guaranteed answer for any single page.


**why i think the rewrite is better.** it says exactly which split i used (client-grouped, not just random), it puts the precision@50 number right next to the base rate instead of alone, and it swaps "predicts" for "ranks by estimated risk" which is more honest about what a model like this can actually promise. its also observational data, nothing was tested with an actual experiment, so i cant say it causes anything, only that this is what i measured.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
